### Q1: Install Spark and PySpark

In [2]:
!java --version

openjdk 17.0.17 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [5]:
pyspark.__file__

'/usr/local/lib/python3.12/dist-packages/pyspark/__init__.py'

In [6]:
spark.version

'4.0.2'

In [7]:
# Скачать файл
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-03 11:19:34--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.10, 13.35.33.83, 13.35.33.60, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   291MB/s    in 0.2s    

2026-03-03 11:19:34 (291 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [8]:
!ls -lh /content/yellow_tripdata_2025-11.parquet

-rw-r--r-- 1 root root 68M Dec 19 15:51 /content/yellow_tripdata_2025-11.parquet


In [9]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



Answer: spark version **4.0.2**

### Q2: Yellow November 2025

In [10]:
df \
    .repartition(4) \
    .write \
    .mode("overwrite") \
    .parquet('data/pq/yellow/2025/11/')

In [ ]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [ ]:
!ls -lh /home/jovyan/work/hw/data/pq/yellow/2025/11/

total 90M
-rw-r--r-- 1 jovyan users 23M Mar  6 15:16 part-00000-a69a19c7-9946-47be-a98d-6769cef764f2-c000.snappy.parquet
-rw-r--r-- 1 jovyan users 23M Mar  6 15:16 part-00001-a69a19c7-9946-47be-a98d-6769cef764f2-c000.snappy.parquet
-rw-r--r-- 1 jovyan users 23M Mar  6 15:16 part-00002-a69a19c7-9946-47be-a98d-6769cef764f2-c000.snappy.parquet
-rw-r--r-- 1 jovyan users 23M Mar  6 15:16 part-00003-a69a19c7-9946-47be-a98d-6769cef764f2-c000.snappy.parquet
-rw-r--r-- 1 jovyan users   0 Mar  6 15:16 _SUCCESS


Answer: **23M**

### Q3: How many taxi trips were there on November 15?

In [11]:
from pyspark.sql import functions as F

In [12]:
df_yellow = spark.read.parquet('data/pq/yellow/2025/11/')

In [13]:
df_yellow.head()

Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 7, 15, 4, 17), tpep_dropoff_datetime=datetime.datetime(2025, 11, 7, 15, 39, 15), passenger_count=1, trip_distance=7.3, RatecodeID=1, store_and_fwd_flag='N', PULocationID=262, DOLocationID=127, payment_type=1, fare_amount=38.7, extra=0.0, mta_tax=0.5, tip_amount=8.54, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=51.24, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.0)

In [14]:
df_yellow \
    .withColumn('pickup_date', F.to_date("tpep_pickup_datetime")) \
    .filter(F.to_date("tpep_pickup_datetime") == F.lit('2025-11-15')) \
    .count()

162604

In [15]:
df_yellow.select("tpep_pickup_datetime", "VendorID", "passenger_count", "trip_distance").show()

+--------------------+--------+---------------+-------------+
|tpep_pickup_datetime|VendorID|passenger_count|trip_distance|
+--------------------+--------+---------------+-------------+
| 2025-11-07 15:04:17|       2|              1|          7.3|
| 2025-11-12 07:48:41|       2|              1|         1.55|
| 2025-11-01 15:24:44|       1|              1|          1.6|
| 2025-11-11 06:14:41|       1|              1|          0.9|
| 2025-11-03 17:34:26|       7|              1|         5.06|
| 2025-11-11 21:47:10|       1|              1|          9.3|
| 2025-11-18 11:47:41|       2|              2|         2.92|
| 2025-11-07 13:57:39|       1|              1|          7.6|
| 2025-11-05 21:06:15|       2|              1|          0.9|
| 2025-11-14 00:41:22|       2|              1|         0.36|
| 2025-11-06 15:58:17|       2|              2|        17.78|
| 2025-11-13 11:31:05|       2|              1|         2.64|
| 2025-11-07 13:22:04|       1|              1|          0.4|
| 2025-1

In [16]:
df_yellow.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [VendorID#20,tpep_pickup_datetime#21,tpep_dropoff_datetime#22,passenger_count#23L,trip_distance#24,RatecodeID#25L,store_and_fwd_flag#26,PULocationID#27,DOLocationID#28,payment_type#29L,fare_amount#30,extra#31,mta_tax#32,tip_amount#33,tolls_amount#34,improvement_surcharge#35,total_amount#36,congestion_surcharge#37,Airport_fee#38,cbd_congestion_fee#39] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/data/pq/yellow/2025/11], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...




In [17]:
df_yellow.createOrReplaceTempView('yellow_tripdata_2025_11')

In [18]:
spark.sql("""
SELECT
    COUNT(1)
FROM
    yellow_tripdata_2025_11
WHERE
    to_date(tpep_pickup_datetime) = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



Answer: **162604**

### Q4: Longest trip for each day

In [19]:
df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [20]:
df_yellow \
    .withColumn(
        'duration',
        (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600) \
    .withColumn(
        'pickup_date',
        F.to_date(F.col('tpep_pickup_datetime'))
    ) \
    .groupBy('pickup_date') \
    .agg(
        F.round(F.max('duration'), 1).alias('max_duration')  # Заміна назви 'max(duration)' на щось більш стандартне
    ) \
    .orderBy(F.col('max_duration').desc()) \
    .limit(5) \
    .show()

+-----------+------------+
|pickup_date|max_duration|
+-----------+------------+
| 2025-11-26|        90.6|
| 2025-11-27|        76.9|
| 2025-11-03|        76.2|
| 2025-11-07|        69.3|
| 2025-11-18|        67.1|
+-----------+------------+



In [21]:
spark.sql("""
SELECT
    to_date(tpep_pickup_datetime) AS pickup_date,
    round(MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600), 1) AS duration
FROM
    yellow_tripdata_2025_11
GROUP BY
    to_date(tpep_pickup_datetime)
ORDER BY
    duration DESC
LIMIT 5;
""").show()

+-----------+--------+
|pickup_date|duration|
+-----------+--------+
| 2025-11-26|    90.6|
| 2025-11-27|    76.9|
| 2025-11-03|    76.2|
| 2025-11-07|    69.3|
| 2025-11-18|    67.1|
+-----------+--------+



Answer Q4: 90.6 hours

## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- 4040
- 8080

Answer Q5: **4040**

### Q6: Least frequent pickup location zone

In [22]:
# Скачать файл
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-03 11:40:18--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.98, 13.35.33.83, 13.35.33.10, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.98|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-03 11:40:18 (58.5 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [23]:
# Показать первые строки файла
!head taxi_zone_lookup.csv

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"
5,"Staten Island","Arden Heights","Boro Zone"
6,"Staten Island","Arrochar/Fort Wadsworth","Boro Zone"
7,"Queens","Astoria","Boro Zone"
8,"Queens","Astoria Park","Boro Zone"
9,"Queens","Auburndale","Boro Zone"


In [24]:
# Прочитать файл в DataFrame с использованием Spark
df1 = spark.read \
.option("header", "true") \
.csv('taxi_zone_lookup.csv')

# Показать уникальные значения в столбце 'Zone'
df1.select('Zone').distinct().show()

# Показать содержимое DataFrame
df1.show()

# Подсчитать количество строк в DataFrame
df1.count()

+--------------------+
|                Zone|
+--------------------+
|Governor's Island...|
|           Homecrest|
|              Corona|
|    Bensonhurst West|
|         Westerleigh|
|      Newark Airport|
|Charleston/Totten...|
|          Douglaston|
|East Concourse/Co...|
|          Mount Hope|
|      Pelham Parkway|
|         Marble Hill|
|           Rego Park|
|       Dyker Heights|
|Heartland Village...|
|Upper East Side S...|
|   Kew Gardens Hills|
|       Rikers Island|
|             Bayside|
|     Jackson Heights|
+--------------------+
only showing top 20 rows
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zo

265

In [25]:
# Записать DataFrame в формате parquet
df1.write.parquet('zones')

In [26]:
!ls -lh zones

total 8.0K
-rw-r--r-- 1 root root 5.9K Mar  3 11:40 part-00000-f051132e-093b-432d-b90c-187076dfde03-c000.snappy.parquet
-rw-r--r-- 1 root root    0 Mar  3 11:40 _SUCCESS


In [27]:
df_zones = spark.read.parquet('zones')

df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [28]:
df_zones.createOrReplaceTempView('zones')

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [31]:
spark.sql("""
SELECT
    zones.Zone AS zone,
    COUNT(1) AS trip_count
FROM
    yellow_tripdata_2025_11
JOIN
    zones
ON
    yellow_tripdata_2025_11.PULocationID = zones.LocationID
GROUP BY
        zone
ORDER BY
        trip_count
LIMIT 5;
""").show()

+--------------------+----------+
|                zone|trip_count|
+--------------------+----------+
|Eltingville/Annad...|         1|
|Governor's Island...|         1|
|       Arden Heights|         1|
|       Port Richmond|         3|
|       Rikers Island|         4|
+--------------------+----------+



Answer Q6: